# Part 2. Metric Engineering and Calculation Validity Logic

## Part Scope

This layer converts normalized annual records into a structured metric package for downstream interpretation.

The role of this layer is not limited to numeric calculation.  
It also defines whether each metric is interpretable, which reporting window supports the calculation, and how calculation status should be preserved for later layers.

The output is therefore not just a set of KPI values, but a metric object with validity context.

## Metric Layer Design

The metric layer starts from normalized annual snapshots rather than raw source responses.

This separation keeps source extraction concerns outside the analytical layer and allows calculations to operate on a stable annual structure.  
The resulting metric package is organized around five analytical views: long-term growth, profitability, cash conversion quality, balance-sheet safety, and recent directional change.

In [ ]:
st.session_state.snapshots = fetch_10k_snapshots(ticker, count=count)

st.session_state.growth = compute_growth_metrics(st.session_state.snapshots)
st.session_state.yoy_list = compute_yoy_growth(st.session_state.snapshots)

latest = _latest_snapshot(st.session_state.snapshots)
st.session_state.valuation = compute_valuation(latest, st.session_state.price_info["close"])
st.session_state.rules = simple_rules(
    st.session_state.valuation,
    st.session_state.growth,
    language=language,
)

## Long-Term Growth Logic

The first metric group measures long-term directional change across revenue, net income, free cash flow, EPS, and BVPS.

For each metric, the workflow searches for the earliest and latest valid annual values rather than assuming a fixed comparison window.  
CAGR is calculated only when the starting value, ending value, and year span remain interpretable.

This rule prevents unstable or structurally invalid growth estimates from being treated as normal outputs.

In [ ]:
def _cagr(start: Optional[float], end: Optional[float], years: int) -> Optional[float]:
    if start is None or end is None or years <= 0:
        return None
    if start <= 0:
        return None
    return (end / start) ** (1 / years) - 1


def _cagr_from_snapshots(
    snaps: List[Any],
    getter: Callable[[Any], Optional[float]],
) -> tuple[Optional[float], tuple[Optional[int], Optional[int]], Dict[str, Any]]:
    start_val, end_val, start_year, end_year = _first_last_valid_pair(snaps, getter)

    if start_year is None or end_year is None:
        return None, (start_year, end_year), {
            "calculable": False,
            "reason": "missing_start_or_end",
            "start_year": start_year,
            "end_year": end_year,
            "span_years": None,
        }

    span_years = end_year - start_year
    if span_years <= 0:
        return None, (start_year, end_year), {
            "calculable": False,
            "reason": "non_positive_year_span",
            "start_year": start_year,
            "end_year": end_year,
            "span_years": span_years,
        }

    cagr_value = _cagr(start_val, end_val, span_years)
    calculable = cagr_value is not None

    return cagr_value, (start_year, end_year), {
        "calculable": calculable,
        "reason": "ok" if calculable else "invalid_start_value_or_data",
        "start_year": start_year,
        "end_year": end_year,
        "span_years": span_years,
    }

## Profitability and Financial Structure

Long-term growth alone is not sufficient to describe operating quality.

For that reason, the workflow adds a second metric group covering three areas:
- margin structure
- cash conversion quality
- balance-sheet safety

This layer tracks net margin, operating margin, and free-cash-flow margin as profitability measures, OCF-to-net-income and FCF-to-net-income as conversion checks, and current ratio, debt-to-equity, and net cash as financial structure indicators.

## Calculation Metadata

Each metric output carries more than a numeric value.

The workflow also stores calculation metadata describing whether the metric was calculable, which years were used, how long the comparison span was, and what limitation blocked calculation when the metric could not be produced.  
This structure allows downstream components to separate weak performance from structurally missing or non-interpretable inputs.

In [ ]:
result = {
    "rev_cagr": rev_cagr,
    "ni_cagr": ni_cagr,
    "fcf_cagr": fcf_cagr,
    "eps_cagr": eps_cagr,
    "bvps_cagr": bvps_cagr,

    "rev_cagr_years": rev_years,
    "ni_cagr_years": ni_years,
    "fcf_cagr_years": fcf_years,
    "eps_cagr_years": eps_years,
    "bvps_cagr_years": bvps_years,

    "net_margin_start": net_m_start,
    "net_margin_end": net_m_end,
    "op_margin_start": op_m_start,
    "op_margin_end": op_m_end,
    "fcf_margin_start": fcf_m_start,
    "fcf_margin_end": fcf_m_end,

    "ocf_to_ni_latest": ocf_ni_ratio,
    "fcf_to_ni_latest": fcf_ni_ratio,
    "current_ratio_latest": current_ratio,
    "debt_to_equity_latest": debt_to_equity,
    "net_cash_latest": net_cash,

    "kpi_metadata": {
        "rev_cagr": rev_meta,
        "ni_cagr": ni_meta,
        "fcf_cagr": fcf_meta,
        "eps_cagr": eps_meta,
        "bvps_cagr": bvps_meta,
        "net_margin": net_margin_meta,
        "op_margin": op_margin_meta,
        "fcf_margin": fcf_margin_meta,
        "quality": quality_meta,
        "safety": safety_meta,
    },

    "calculation_flags": {
        "rev_cagr_calculable": rev_meta["calculable"],
        "ni_cagr_calculable": ni_meta["calculable"],
        "fcf_cagr_calculable": fcf_meta["calculable"],
        "eps_cagr_calculable": eps_meta["calculable"],
        "bvps_cagr_calculable": bvps_meta["calculable"],
        "net_margin_calculable": net_margin_meta["calculable"],
        "op_margin_calculable": op_margin_meta["calculable"],
        "fcf_margin_calculable": fcf_margin_meta["calculable"],
        "ocf_to_ni_calculable": quality_meta["ocf_to_ni_calculable"],
        "fcf_to_ni_calculable": quality_meta["fcf_to_ni_calculable"],
        "current_ratio_calculable": safety_meta["current_ratio_calculable"],
        "debt_to_equity_calculable": safety_meta["debt_to_equity_calculable"],
        "net_cash_calculable": safety_meta["net_cash_calculable"],
    },
}

## Recent Change Layer

Long-term metrics are complemented by a separate year-over-year layer.

Where CAGR captures structural change across a broader comparison window, the Y/Y layer isolates recent directional movement between adjacent fiscal years.  
This separation preserves the distinction between long-term trajectory and short-term momentum.

In [ ]:
def compute_yoy_growth(snapshots: List[Any]) -> List[Dict[str, Any]]:
    snaps = _sorted_snaps(snapshots)
    if len(snaps) < 2:
        return []

    def growth(prev_val: Optional[float], curr_val: Optional[float]) -> Optional[float]:
        if prev_val is None or curr_val is None or prev_val == 0:
            return None
        return (curr_val - prev_val) / prev_val

    results: List[Dict[str, Any]] = []

    for i in range(1, len(snaps)):
        prev = snaps[i - 1]
        curr = snaps[i]

        results.append({
            "year_from": prev.fiscal_year,
            "year_to": curr.fiscal_year,
            "rev_growth": growth(getattr(prev, "revenue", None), getattr(curr, "revenue", None)),
            "ni_growth": growth(getattr(prev, "net_income", None), getattr(curr, "net_income", None)),
            "fcf_growth": growth(_compute_fcf(prev), _compute_fcf(curr)),
            "eps_growth": growth(_compute_eps(prev), _compute_eps(curr)),
            "bvps_growth": growth(_compute_bvps(prev), _compute_bvps(curr)),
        })

    return results

## Downstream Readiness

At the end of this layer, the workflow no longer holds only normalized annual records.

It now holds a structured analytical package composed of metric values, comparison windows, calculation flags, and diagnostic metadata.  
That package becomes the direct input to the interpretation layer, where numerical outputs are reorganized into rule-based summaries and narrative-ready structures.

## Transition to the Next Layer

With the metric layer in place, the workflow can move beyond numerical transformation.

The next section covers how these derived values are reorganized into structured diagnostic objects, threshold-based summaries, and narrative-ready interpretation inputs.